<a href="https://colab.research.google.com/github/Kurasce/JaiTTS_JAKK_CLONE/blob/main/JaiTTS_F5TTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JaiTTS-F5TTS: Thai Voice Cloning (Colab)

Run inference with [JaiTTS-F5TTS](https://huggingface.co/JTS-AI/JaiTTS-F5TTS), a non-autoregressive Thai zero-shot voice cloning model based on F5-TTS.

- Paper: *JaiTTS: A Thai Voice Cloning Model* ([arXiv:2604.27607](https://arxiv.org/abs/2604.27607))
- Model: [huggingface.co/JTS-AI/JaiTTS-F5TTS](https://huggingface.co/JTS-AI/JaiTTS-F5TTS)
- Inference codebase adapted from [ThonburianTTS](https://github.com/biodatlab/thonburian-tts)

**Research prototype** — released for research and benchmarking only.

**Before you start:** In Colab, go to `Runtime > Change runtime type` and select a **GPU** (e.g. T4) for reasonable inference speed.

## 0. Check GPU

In [ ]:
!nvidia-smi

Thu Aug  6 10:20:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the inference codebase
This model uses the `flowtts` pipeline adapted from ThonburianTTS.

In [ ]:
import os

if not os.path.isdir("thonburian-tts"):
    !git clone https://github.com/biodatlab/thonburian-tts.git

%cd thonburian-tts

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

Cloning into 'thonburian-tts'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 214 (delta 66), reused 19 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (214/214), 2.77 MiB | 11.01 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/thonburian-tts


## 2. Install dependencies
Installs from the repo's own `requirements.txt` (torch, f5-tts, pydub, vocos, pythainlp, etc.) so versions stay in sync with the `flowtts` pipeline code.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 115.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q python-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 6.2 MB/s eta 0:00:00


## 3. Upload a reference audio clip

Upload a short `.wav` (or other audio) file containing the voice you want to clone, along with an accurate transcription of what is said in it.

In [ ]:
from google.colab import files

uploaded = files.upload()
reference_audio_path = next(iter(uploaded.keys()))
print(f"Using reference audio: {reference_audio_path}")

Saving auido.m4a to auido.m4a
Using reference audio: auido.m4a


## 4. Set reference transcription and text to generate
Edit the strings below. Leave `reference_text` as an empty string `""` to let the pipeline auto-transcribe your reference audio instead.

In [ ]:
reference_text = "ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ "  # @param {type:"string"}
gen_text = "ถ้าคุณขายของออนไลน์ แล้วเคยต้องนั่งไล่หาคลิปว่าออเดอร์นี้แพ็กอะไรไปบ้าง  นี่คือสิ่งที่ผมกำลังทำครับ — ProofMe  แค่สแกน Tracking หรือ Order Number  แล้วเริ่มแพ็ก ProofMe จะบันทึกวิดีโอ ผูกคลิปกับออเดอร์  และทำให้ค้นหาหลักฐานย้อนหลังได้ทันที  ไม่ต้องตั้งชื่อไฟล์ ไม่ต้องเปิดคอม ไม่ต้องไล่หาคลิปเป็นร้อย ๆ ไฟล์  ทั้งหมดจบได้ในโทรศัพท์เครื่องเดียว"  # @param {type:"string"}

### 4b. Double Space Bar Text Chunking
เราจะแบ่งข้อความเป็นชิ้นๆ โดยใช้ `  ` (double space) เป็นตัวคั่น เพื่อให้แต่ละชิ้นมีความยาวเหมาะสมสำหรับการสังเคราะห์เสียง.

In [ ]:
def get_double_space_chunks(text):
    # Split by double space, keeping only non-empty strings
    return [chunk.strip() for chunk in text.split('  ') if chunk.strip()]

# Process the chunks using double space bar
print("Splitting text by double space bar...")
sentences_to_process = get_double_space_chunks(gen_text)

print(f"Text split into {len(sentences_to_process)} chunks:")
for i, s in enumerate(sentences_to_process):
    print(f"{i+1}: {s}")

Splitting text by double space bar...
Text split into 7 chunks:
1: ถ้าคุณขายของออนไลน์ แล้วเคยต้องนั่งไล่หาคลิปว่าออเดอร์นี้แพ็กอะไรไปบ้าง
2: นี่คือสิ่งที่ผมกำลังทำครับ — ProofMe
3: แค่สแกน Tracking หรือ Order Number
4: แล้วเริ่มแพ็ก ProofMe จะบันทึกวิดีโอ ผูกคลิปกับออเดอร์
5: และทำให้ค้นหาหลักฐานย้อนหลังได้ทันที
6: ไม่ต้องตั้งชื่อไฟล์ ไม่ต้องเปิดคอม ไม่ต้องไล่หาคลิปเป็นร้อย ๆ ไฟล์
7: ทั้งหมดจบได้ในโทรศัพท์เครื่องเดียว


หลังจากรันด้านบนแล้ว ใน Cell ถัดไป (Step 6) จะใช้ `sentences_to_process` ที่ถูกแบ่งมาแล้ว.

## 5. Load the JaiTTS-F5TTS pipeline

In [ ]:
import torch
from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig

model_config = ModelConfig(
    language="th",
    model_type="F5",
    checkpoint="hf://JTS-AI/JaiTTS-F5TTS/model.pt",
    vocab_file="hf://JTS-AI/JaiTTS-F5TTS/vocab.txt",
    vocoder="vocos",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

audio_config = AudioConfig(
    silence_threshold=-45,
    cfg_strength=2.0, # Reduced slightly for stability
    nfe_step=32,
    speed=1.0,
)

# Initialize pipeline with explicit silence removal to avoid tensor mismatch
pipeline = FlowTTSPipeline(model_config=model_config, audio_config=audio_config)
# Force internal model to remove silence from reference which often fixes the tensor mismatch
pipeline.model.remove_silence = True

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /root/.cache/huggingface/hub/models--JTS-AI--JaiTTS-F5TTS/snapshots/651e39bf13b2475ca5f381234128ed97727f8b7a/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--JTS-AI--JaiTTS-F5TTS/snapshots/651e39bf13b2475ca5f381234128ed97727f8b7a/model.pt 



## 6. Generate speech

In [ ]:
import re
from pathlib import Path
from pydub import AudioSegment

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Ensure the temp directory used by the pipeline exists
Path("temp").mkdir(parents=True, exist_ok=True)

# Provide reference_text explicitly (step 4) if you can
ref_text = reference_text.strip() or None

# Use the chunked sentences from the previous step
print(f"Using {len(sentences_to_process)} chunks for speech generation.")

chunk_paths = []
for i, sentence in enumerate(sentences_to_process):
    print(f"Generating chunk {i+1}/{len(sentences_to_process)}: {sentence[:30]}...")
    chunk_path = pipeline(
        text=sentence,
        ref_voice=reference_audio_path,
        ref_text=ref_text,
        output_file=str(output_dir / f"chunk_{i:03d}.wav"),
        speed=audio_config.speed,
        check_duration=True,
    )
    chunk_paths.append(chunk_path)

# Stitch the per-sentence clips back together with a short pause between them.
gap = AudioSegment.silent(duration=150)
combined = AudioSegment.silent(duration=0)
for i, path in enumerate(chunk_paths):
    combined += AudioSegment.from_wav(path)
    if i < len(chunk_paths) - 1:
        combined += gap

output_path = str(output_dir / "output.wav")
combined.export(output_path, format="wav")
print(f"\nSaved combined audio to {output_path}")

Using 7 chunks for speech generation.
Generating chunk 1/7: ถ้าคุณขายของออนไลน์ แล้วเคยต้อ...
Converting audio...
Using custom reference text...

ref_text   ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ. 
gen_text 0 ถ้าคุณขายของออนไลน์ แล้วเคยต้องนั่งไล่หาคลิปว่าออเดอร์นี้แพ็กอะไรไปบ้าง


Generating audio in 1 batches...


100%|██████████| 1/1 [00:03<00:00,  3.77s/it]


Time taken is 4.27 seconds
Generating chunk 2/7: นี่คือสิ่งที่ผมกำลังทำครับ — P...
Converting audio...
Using custom reference text...

ref_text   ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ. 
gen_text 0 นี่คือสิ่งที่ผมกำลังทำครับ — ProofMe


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


Time taken is 3.13 seconds
Generating chunk 3/7: แค่สแกน Tracking หรือ Order Nu...
Converting audio...
Using custom reference text...

ref_text   ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ. 
gen_text 0 แค่สแกน Tracking หรือ Order Number


Generating audio in 1 batches...


...

Time taken is 3.26 seconds
Generating chunk 6/7: ไม่ต้องตั้งชื่อไฟล์ ไม่ต้องเปิ...
Converting audio...
Using custom reference text...

ref_text   ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ. 
gen_text 0 ไม่ต้องตั้งชื่อไฟล์ ไม่ต้องเปิดคอม ไม่ต้องไล่หาคลิปเป็นร้อย ๆ ไฟล์


Generating audio in 1 batches...


100%|██████████| 1/1 [00:03<00:00,  3.40s/it]


Time taken is 3.70 seconds
Generating chunk 7/7: ทั้งหมดจบได้ในโทรศัพท์เครื่องเ...
Converting audio...
Using custom reference text...

ref_text   ใครที่ขายของออนไลน์ ถ้าใครมีปัญหานะครับ เรื่องของการถ่ายวิดีโอ เก็บหลักฐานตอนแพ็กสินค้าเนี่ย ผมมีโซลูชันให้นะครับ. 
gen_text 0 ทั้งหมดจบได้ในโทรศัพท์เครื่องเดียว


Generating audio in 1 batches...


100%|██████████| 1/1 [00:02<00:00,  2.95s/it]

Time taken is 3.27 seconds

Saved combined audio to outputs/output.wav


## 7. Play and download the result

In [ ]:
from IPython.display import Audio, display

display(Audio(output_path))

In [ ]:
from google.colab import files

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Citation
```
@misc{karnjanaekarin2026jaittsthaivoicecloning,
      title={JaiTTS: A Thai Voice Cloning Model},
      author={Jullajak Karnjanaekarin and Pontakorn Trakuekul and Narongkorn Panitsrisit and Sumana Sumanakul and Vichayuth Nitayasomboon and Nithid Guntasin and Thanavin Denkavin and Attapol T. Rutherford},
      year={2026},
      eprint={2604.27607},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2604.27607},
}
```
Codebase adapted from [ThonburianTTS](https://github.com/biodatlab/thonburian-tts).